<a href="https://colab.research.google.com/github/Praneethgajula01/aml-0026/blob/main/wrapper_methods_aml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.datasets import load_breast_cancer #load_diabetes
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression #lassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

data=load_breast_cancer()
X=data.data
y=data.target

print(X.shape)


(569, 30)


Forward Feature Selection

In [ ]:
model=Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=5000))
])

selector=SequentialFeatureSelector(
    model,
    n_features_to_select=5,
    direction="forward",
    scoring="accuracy",
    cv=5
)

selector.fit(X, y)
print(data.feature_names[selector.get_support()])

['mean symmetry' 'concavity error' 'worst texture' 'worst perimeter'
 'worst smoothness']


Backward Feature Elimination

In [ ]:
selector=SequentialFeatureSelector(
    model,
    n_features_to_select=5,
    direction="backward",
    scoring="accuracy",
    cv=5
)

selector.fit(X, y)
print(data.feature_names[selector.get_support()])

['mean texture' 'mean concave points' 'worst area' 'worst smoothness'
 'worst concavity']


Recursive Feature Elimination

In [ ]:
from sklearn.feature_selection import RFE

data=load_breast_cancer()
X=data.data
y=data.target


X_scaled=StandardScaler().fit_transform(X)

model=LogisticRegression(max_iter=5000)

selector=RFE(
    estimator=model,
    n_features_to_select=5,
    step=1
)

selector.fit(X_scaled, y)

for name, selected, rank in zip(
    data.feature_names,
    selector.support_,
    selector.ranking_
):
    print(name, selected, rank)

mean radius False 12
mean texture False 14
mean perimeter False 15
mean area False 8
mean smoothness False 23
mean compactness False 10
mean concavity False 13
mean concave points False 3
mean symmetry False 25
mean fractal dimension False 20
radius error True 1
texture error False 22
perimeter error False 9
area error False 4
smoothness error False 18
compactness error False 6
concavity error False 24
concave points error False 21
symmetry error False 17
fractal dimension error False 16
worst radius True 1
worst texture True 1
worst perimeter False 2
worst area True 1
worst smoothness False 7
worst compactness False 26
worst concavity False 5
worst concave points True 1
worst symmetry False 11
worst fractal dimension False 19


recursive feature elemination with cross validation RFECV


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import RFECV

model=LogisticRegression(max_iter=5000)

cv=StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

selector=RFECV(
    estimator=model,
    step=1,
    cv=cv,
    scoring="accuracy",
    min_features_to_select=1
)

selector.fit(X_scaled, y)

print("optimal number of features:", selector.n_features_)

print("selected features:", data.feature_names[selector.support_])

optimal number of features: 23
selected features: ['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean compactness' 'mean concavity' 'mean concave points' 'radius error'
 'perimeter error' 'area error' 'smoothness error' 'compactness error'
 'symmetry error' 'fractal dimension error' 'worst radius' 'worst texture'
 'worst perimeter' 'worst area' 'worst smoothness' 'worst concavity'
 'worst concave points' 'worst symmetry' 'worst fractal dimension']


In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
import pandas as pd

data=load_diabetes()
X=pd.DataFrame(data.data, columns=data.feature_names)
y=data.target

scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)

lasso=LassoCV(
    cv=5,
    random_state=42,
    max_iter=1000
)

lasso.fit(X_scaled, y)

coefficients=pd.Series(
    lasso.coef_,
    index=X.columns
)

selected_features=coefficients[
    coefficients.abs()>1e-6
]

print("beast alpha: ", lasso.alpha_)

print("selected features:", selected_features)

beast alpha:  0.07891843500595844
selected features: age    -0.308921
sex   -11.225046
bmi    24.816849
bp     15.271304
s1    -27.085410
s2     14.386231
s4      6.835041
s5     31.864972
s6      3.179041
dtype: float64


l1 regularized logistic regression


In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

data=load_breast_cancer()
X=data.data
y=data.target

pipeline=Pipeline([
    ("scaler", StandardScaler()),
    ("feature_selection", SelectFromModel(
        LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=0.1,
            max_iter=5000
        )
    )),
    ("classification", LogisticRegression(max_iter=5000))
])

pipeline.fit(X, y)
mask=pipeline.named_steps["feature_selection"].get_support()
print(data.feature_names[mask])

['mean concave points' 'radius error' 'worst radius' 'worst texture'
 'worst smoothness' 'worst concavity' 'worst concave points'
 'worst symmetry']


ensemble

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris

from sklearn.feature_selection import SelectFromModel
iris=load_iris()

X, y=iris.data, iris.target
X.shape

(150, 4)

In [ ]:
clf=RandomForestClassifier(n_estimators=10)
clf=clf.fit(X, y)
clf.feature_importances_





array([0.08548489, 0.02480599, 0.41204997, 0.47765914])

In [ ]:
model=SelectFromModel(clf, prefit=True)
X_new=model.transform(X)
X_new.shape


(150, 2)

feature selection using pipeline

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

from sklearn.pipeline import Pipeline

clfs=Pipeline([("feature_selection", SelectFromModel(RandomForestClassifier(n_estimators=10))),
               ("classifier")
               ])

attendance student data

In [ ]:
import pandas as pd

data={
    "study_hours": [1,2,3,4,5,6,7,8,9],
    "Attendance": [50, 55, 60, 65, 70, 75, 80, 90, 95],
    "pass": [0,0,0,0,0, 1,1,1,1]
}

df=pd.DataFrame(data)

X=df[["study_hours", "Attendance"]]

y=df["pass"]



In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

x_train_model=scaler.fit_transform(X_train)

X_test

,study_hours,Attendance
7,8,90
1,2,55
5,6,75


In [ ]:
X_test_scaled = scaler.transform(X_test)
display(X_test_scaled)

array([[ 1.21386547,  1.38564065],
       [-1.08609016, -1.03923048],
       [ 0.4472136 ,  0.34641016]])

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

prob_positive=model.predict_proba(X_test_scaled)[:, 1]


fpr, tpr, thresholds=roc_curve(y_test, prob_positive)
auc_value=roc_auc_score(y_test, prob_positive)

plt.plot()